# Train + Validate + Calibrate

Uses the modular pipeline (DataModule → Model → Trainer). Edit overrides to swap models/augs/hparams/W&B.


In [ ]:
import sys
from pathlib import Path

# Prefer local src/ over any installed package
sys.path.insert(0, str((Path("..").resolve() / "src")))

In [ ]:
# Choose device manually ("cpu" or "cuda"); set to None to auto-detect
DEVICE_OVERRIDE = None  # e.g., "cpu" to force CPU, "cuda" to force GPU if available

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Plot theming
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 300, 'savefig.dpi': 300, 'axes.titlesize': 12, 'axes.labelsize': 11, 'legend.fontsize': 10})

PLOTS_DIR = Path('../plots')
PLOTS_DIR.mkdir(exist_ok=True)

def save_fig(fig, name: str, prefix: str = "train"):
    path = PLOTS_DIR / f"{prefix}_{name}.pdf"
    fig.savefig(path, bbox_inches='tight', format='pdf')
    print(f"Saved figure to {path}")
    return path

In [ ]:
from config import TrainConfig, build_config_from_dict
from training import Trainer

overrides = {
    "data_root": str(Path("../data")),
    "checkpoints_dir": str(Path("../artifacts/checkpoints")),
    "epochs": 5,  # bump for full runs
    "wandb_mode": "disabled",  # set to 'online' to log to W&B
    "tensorboard": False,  # set True to log TensorBoard summaries to tensorboard_dir
    "progress_bar": True,
    # "model_name": "efficientnet_b0",
    # "horizontal_flip": 0.5,
    # "max_rotation": 15,
}

cfg = build_config_from_dict(overrides)
if DEVICE_OVERRIDE:
    cfg.device_override = DEVICE_OVERRIDE  # Trainer will respect this attr if set
cfg

In [ ]:
trainer = Trainer(cfg)
train_summary = trainer.fit()
test_metrics = trainer.test()
train_summary, test_metrics

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix
from validation import evaluate, reliability_bins

history_df = pd.DataFrame(train_summary["history"])
display(history_df)

# Loss curves
fig, ax = plt.subplots(figsize=(7, 4))
history_df.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", ax=ax)
ax.set_ylabel("Loss")
ax.set_title("Training/Validation Loss")
save_fig(fig, "loss_curves")
plt.show()

# Validation metrics over epochs
metric_cols = [col for col in history_df.columns if col.startswith("val/") and col not in ("val/loss",)]
if metric_cols:
    fig, ax = plt.subplots(figsize=(7, 4))
    history_df.plot(x="epoch", y=metric_cols, marker="o", ax=ax)
    ax.set_ylabel("Metric")
    ax.set_title("Validation Metrics")
    save_fig(fig, "val_metrics")
    plt.show()

# Evaluate on validation for detailed plots
val_metrics, val_outputs = evaluate(
    trainer.model,
    trainer.val_loader,
    trainer.device,
    trainer.cfg,
    temperature=getattr(trainer, "temperature", None),
)

val_probs = val_outputs["probs"]
val_labels = val_outputs["labels"].cpu().numpy()
val_preds = val_probs.argmax(axis=1)
class_names = trainer.datamodule.train_dataset.classes

# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Validation Confusion Matrix")
save_fig(fig, "confusion")
plt.show()

# Score histogram for positive class
pos_scores = val_probs[:, 1] if val_probs.shape[1] > 1 else val_probs[:, 0]
fig, ax = plt.subplots(figsize=(6, 4))
for label_val, label_name in enumerate(class_names):
    ax.hist(pos_scores[val_labels == label_val], bins=20, alpha=0.6, label=label_name)
ax.set_xlabel("Predicted probability (class 1)")
ax.set_ylabel("Count")
ax.set_title("Validation Score Histogram")
ax.legend()
save_fig(fig, "score_hist")
plt.show()

# Reliability diagram
bin_conf, bin_acc, bin_count = reliability_bins(
    val_probs, val_labels, n_bins=trainer.cfg.reliability_bins
)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.bar(bin_conf, bin_acc, width=1.0 / trainer.cfg.reliability_bins, alpha=0.6, align="center")
ax.set_xlabel("Confidence")
ax.set_ylabel("Accuracy")
ax.set_title(f"Validation Reliability | ECE={val_metrics['ece']:.3f}")
ax.legend()
save_fig(fig, "reliability")
plt.show()